# Education based on conversation


- [ ] fix the regex to load the pkl better
- [ ] handle the 7 file failures (bad metadata processing)

Notes on the transcript types
- M01: life map
- N01: field notes
- TL01: time log
- F01: transcription (track)

In [1]:
%pip install gdown pandas numpy seaborn matplotlib PyPDF2 openpyxl

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Import files using a zip folder instead of mounting drive for security

import gdown
import zipfile

file_id = '1hQeLbh7CioPw8tD8HrTAjuJfSt00JAvh'       # file ID from share link
output_filename = 'raw_interview_transcripts.zip'

try:
    gdown.download(id=file_id, output=output_filename, quiet=False)
    print(f"File '{output_filename}' downloaded successfully.")
except Exception as e:
    print(f"An error occurred during download: {e}")

with zipfile.ZipFile('raw_interview_transcripts.zip', 'r') as zip_ref:
    zip_ref.extractall()


Downloading...
From (original): https://drive.google.com/uc?id=1hQeLbh7CioPw8tD8HrTAjuJfSt00JAvh
From (redirected): https://drive.google.com/uc?id=1hQeLbh7CioPw8tD8HrTAjuJfSt00JAvh&confirm=t&uuid=2ca93e92-c957-48de-9b37-461b576ef512
To: /Users/rlay/work/26s-curric-803-report/raw_interview_transcripts.zip
100%|██████████| 120M/120M [00:03<00:00, 38.4MB/s] 


File 'raw_interview_transcripts.zip' downloaded successfully.


In [3]:
# datascience libs
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# file handling libs
import os
import PyPDF2
import re

In [4]:
# load pdf file names into a list, only including F01 (transcription notes)
pdf_dir = './individual_transcripts/'

pdf_files = [os.path.join(pdf_dir, f) for f in os.listdir(pdf_dir) if f.lower().endswith('.pdf') and "F01" in f]

# verify files loaded
print(f"Found {len(pdf_files)} PDF files:")
for pdf_file in pdf_files[:10]:
    print(pdf_file)


Found 250 PDF files:
./individual_transcripts/VAOHP0101_F01.pdf
./individual_transcripts/VAHF0002_F01.pdf
./individual_transcripts/VAOHP0111_F01.pdf
./individual_transcripts/VAOHP0069_F01.pdf
./individual_transcripts/VAHF0010_F01_Eng.pdf
./individual_transcripts/VAHF0012_F01_Viet.pdf
./individual_transcripts/VAOHP0100_F01_Eng.pdf
./individual_transcripts/VAOHP0058_F01_Viet.pdf
./individual_transcripts/VAOHP0215_F01.pdf
./individual_transcripts/VAOHP0074.2_F01_Eng.pdf


In [5]:
def extract_text_from_pdf(pdf_path):
    """
    Extracts all text content from a given PDF file.

    Args:
        pdf_path (str): The path to the PDF file.

    Returns:
        str: The extracted text content, or an empty string if an error occurs.
    """
    text_content = ''
    try:
        with open(pdf_path, 'rb') as file:
            reader = PyPDF2.PdfReader(file)
            for page_num in range(len(reader.pages)):
                page = reader.pages[page_num]
                text_content += page.extract_text() + '\n'
        return text_content
    except FileNotFoundError:
        print(f"Error: File not found at {pdf_path}")
        return ""
    except Exception as e:
        print(f"Error extracting text from '{pdf_path}': {e}")
        return ""

In [6]:
# verify files have contents
first_pdf_file = pdf_files[0]
if first_pdf_file:
    first_pdf_extracted_text = extract_text_from_pdf(first_pdf_file)
    if first_pdf_extracted_text:
        print(f"Successfully extracted text from '{first_pdf_file}' using the function.\nFirst 500 characters of text:\n")
        display(f"{first_pdf_extracted_text[:500]}...")
    else:
        print(f"Failed to extract text from '{first_pdf_file}' using the function.")
else:
    print("No PDF files found to test the function with.")


Successfully extracted text from './individual_transcripts/VAOHP0101_F01.pdf' using the function.
First 500 characters of text:



'Vietnamese American Oral History Project, UC Irvine   Narrator: MARY HOANG LONG Interviewer: Nina Mai Thi Long Date: November 13, 2012 Location: Westminster, California Sub-Collection: Vietnamese American Experience Course, Fall 2012 Length of Interview: 01:32:16 NL: Today is Tuesday, November 13, 2012. This is Nina Long with the Vietnamese American Oral History Project and I am interviewing Ms. Mary Long. We are at her home in Westminster, California.               NL: Hi Ms. Long              ...'

In [ ]:
columns = [
    # hard metadata
    'interview_id',
    'narrator_name',
    'narrator_initials',
    'interviewer_name',
    'interviewer_initials',
    'date',
    'location',
    'language',
    'interview_length',
    'sub-collection',
    'narrator_dialogue',
    'interviewer_dialogue',
    # inferred metadata
    'wave',
    'generation',
    'gender',
    'age',
    'birth_year',
    'birthplace',
    'year_left_country',
    'year_of_arrival',
    'refugee_cohort',
    'highest_education',
]
df = pd.DataFrame(columns=columns)
print("Empty DataFrame created with specified columns:")
print(df.head())

Empty DataFrame created with specified columns:
Empty DataFrame
Columns: [interview_id, narrator_name, narrator_initials, interviewer_name, interviewer_initials, date, location, language, interview_length, sub-collection, narrator_dialogue, interviewer_dialogue]
Index: []


In [8]:
def _remove_filename_from_text(text_content, file_name):
    """
    Removes the file name from the raw text content of the PDF and detects language
    
    Args:
        text_content (str): The raw text content extracted from a PDF.
        file_name (str): The name of the PDF file.

    Returns:
        tuple: A tuple containing (cleaned_text_content, language)
    """

    # Detect language based on filename patterns
    if 'viet' in file_name.lower():
        language = 'Vietnamese'
    elif 'cantonese' in file_name.lower():
        language = 'Cantonese'
    else:
        language = 'English'
    
    # trim file_name to remove directory and file extensions
    # r'/([^\/]+?)_F01' regex to select specifically the part between rawcontent/{target}_F01
    file_name = file_name.replace(f"{os.path.dirname(file_name)}/", '')
    file_name = file_name.replace('_F01.pdf', '')
    file_name = file_name.replace('_F01.docx.pdf', '')
    file_name = re.sub(r'/([^\/]+?)_F01', '', file_name)
    text_content = re.sub(fr"{file_name}\d+", "", text_content)
    
    return text_content, language

# test
cleaned_text, detected_language = _remove_filename_from_text(first_pdf_extracted_text, pdf_files[7])
display(cleaned_text)
display(f"Detected language: {detected_language}")
display(pdf_files[7])

'Vietnamese American Oral History Project, UC Irvine   Narrator: MARY HOANG LONG Interviewer: Nina Mai Thi Long Date: November 13, 2012 Location: Westminster, California Sub-Collection: Vietnamese American Experience Course, Fall 2012 Length of Interview: 01:32:16 NL: Today is Tuesday, November 13, 2012. This is Nina Long with the Vietnamese American Oral History Project and I am interviewing Ms. Mary Long. We are at her home in Westminster, California.               NL: Hi Ms. Long              MHL: Hi Nina.                 NL: What is your full name?             MHL: My full name is Mary Hoang Long.             NL: And how old are you right now?            MHL: I am 46 years old.               NL: And when is your date of birth?            MHL: My date of birth is April 6, 1966.             NL: And what is your place of birth?           MHL: I was born in Vietnam and in Saigon City.            NL: What are your parents\' names?           MHL: My father name is True Hoang and my mom\'

'Detected language: Vietnamese'

'./individual_transcripts/VAOHP0058_F01_Viet.pdf'

In [ ]:
def _initialize_metadata():
    """
    Initializes the metadata dictionary with None values.
    """
    return {
        # read metadata
        'interview_id': None,
        'narrator_name': None,
        'interviewer_name': None,
        'date': None,
        'location': None,
        'language': None,
        'interview_length': None,
        'sub-collection': None,
        'narrator_initials': None,
        'interviewer_initials': None,
        'narrator_dialogue': None,
        'interviewer_dialogue': None,
        # inferred metadata
        'wave': None,
        'generation': None,
        'gender': None,
        'age': None,
        'birth_year': None,
        'birthplace': None,
        'year_left_country': None,
        'year_of_arrival': None,
        'refugee_cohort': None,
        'highest_education': None,
    }

In [10]:
def _get_metadata_patterns():
    """
    Defines and returns regular expression patterns for each metadata field.
    """
    return {
        'narrator_name': r"Narrator:([\s\S]*?)(?:Interviewer:|Date:|Location:|Sub-Collection:|Length of Interview:|$)",
        'interviewer_name': r"Interviewer:([\s\S]*?)(?:Date:|Location:|Sub-Collection:|Length of Interview:|$)",
        'date': r"Date:([\s\S]*?)(?:Location:|Sub-Collection:|Length of Interview:|$)",
        'location': r"Location:([\s\S]*?)(?:Sub-Collection:|Length of Interview:|$)",
        'interview_length': r"Length of Interview:\s+(\d{2}:\d{2}:\d{2})",
        'sub-collection': r"Sub-Collection:([\s\S]*?)(?:Length of Interview:|$)"
    }

In [11]:
def _extract_initial_fields(text_content, patterns):
    """
    Extracts initial metadata fields from the given text content using the provided patterns.
    """
    metadata = _initialize_metadata()
    for key, pattern in patterns.items():
        match = re.search(pattern, text_content, re.IGNORECASE | re.DOTALL)
        if match:
            metadata[key] = match.group(1).strip()
    return metadata

In [12]:
def _refine_name_field(name_string):
    """
    Refines narrator_name or interviewer_name by truncating extraneous text.
    """
    return name_string.lower().title()

In [13]:
def _clean_all_values(data):
    """
    Cleans up residual newlines or extra spaces in all extracted string values in a dictionary.
    """
    for key, value in data.items():
        if value and isinstance(value, str):
            data[key] = re.sub(r'\s+', ' ', value).strip()
    return data

In [62]:
def _extract_dialogue_section(text_content):
    """
    Extracts the dialogue section of the transcript.
    If not found, it tries to find the start of the first speaker turn.

    Args:
        text_content (str): The full text content of the PDF.

    Returns:
        str: The dialogue section, or an empty string if no clear dialogue start is found.
    """
    # This assumes dialogue starts with a speaker's initials (e.g., NDM:, PCN:).
    # The regex grabs any 2+ uppercase letters followed by a colon
    speaker_initials_pattern = re.compile(r'[A-Z]{2,}:', re.DOTALL)
    match = speaker_initials_pattern.search(text_content)
    if match:
        # Start dialogue from the beginning of the first identified speaker initial
        return text_content[match.start():].strip()
    else:
        # If neither initials are found, we cannot reliably determine
        # the start of the dialogue, so return an empty string.
        return ""

In [63]:
def _extract_actual_speaker_initials(text_content):
    """
    Parses the dialogue section to identify the interviewer's initials
    (the first initial found) and the narrator's initials (the second distinct initial found).

    Args:
        text_content (str): The full text content of the PDF.

    Returns:
        tuple: A tuple containing (narrator_initials, interviewer_initials).
    """
    narrator_initials = None
    interviewer_initials = None

    dialogue_section_full = _extract_dialogue_section(text_content)

    if not dialogue_section_full:
        return (None, None)

    speaker_initials_pattern = re.compile(r'\b([A-Z]{2,5})\s*:')
    matches = list(speaker_initials_pattern.finditer(dialogue_section_full))

    # Collect all unique speaker initials
    unique_speakers = []
    for match in matches:
        current_initials = match.group(1)
        if current_initials not in unique_speakers:
            unique_speakers.append(current_initials)

    # Assign roles based on order and count
    if len(unique_speakers) >= 1:
        interviewer_initials = unique_speakers[0]  # First speaker is interviewer
    if len(unique_speakers) >= 2:
        narrator_initials = unique_speakers[1]     # Second speaker is primary narrator

    return (narrator_initials, interviewer_initials)

In [79]:
def _parse_speakers_dialogue(dialogue_text, narrator_initials, interviewer_initials):
    """
    Parses the dialogue text and attributes lines to narrator and interviewer
    based on their initials.

    Args:
        dialogue_text (str): The extracted dialogue section of the transcript.
        narrator_initials (str): The initials of the narrator.
        interviewer_initials (str): The initials of the interviewer.

    Returns:
        tuple: A tuple containing (narrator_dialogue_list, interviewer_dialogue_list),
               where each is a list of their respective dialogue segments.
    """
    narrator_dialogue_list = []
    interviewer_dialogue_list = []

    if not narrator_initials or not interviewer_initials:
        raise ValueError(f"failing due to bad initials {narrator_initials = }\t{interviewer_initials = }")

    # Clean non-breaking spaces from dialogue text
    dialogue_text = dialogue_text.replace('\xa0', ' ')
    dialogue_text = dialogue_text.replace('\t\r', '')

    narrator_init_esc = re.escape(narrator_initials)
    interviewer_init_esc = re.escape(interviewer_initials)

    # Create speaker pattern for all speakers
    speaker_pattern = f"{narrator_init_esc}|{interviewer_init_esc}"

    # Regex to find speaker turns: (SpeakerInitials:)(DialogueText)
    # The non-greedy match (.*?) captures dialogue until the next speaker initial or end of string.
    # The lookahead `(?= ... |$)` ensures that the delimiter itself is not consumed by the match,
    # allowing subsequent matches to start correctly.
    speaker_turn_pattern = re.compile(
        rf"([A-Z]{{2,}}:)(.*?)(?=([A-Z]{{2,}}:)|$)",
        # rf"({speaker_pattern}:)(.*?)(?=(?:{speaker_pattern}:)|$)",
        
        re.MULTILINE
    )

    for match in speaker_turn_pattern.finditer(dialogue_text):
        speaker_id = match.group(1).strip() # e.g., "NL:", "MHL:"
        dialogue_segment = match.group(2).strip()

        if speaker_id == f"{interviewer_initials}:":
            interviewer_dialogue_list.append(dialogue_segment)
        else:
            narrator_dialogue_list.append(dialogue_segment)
    
    # Convert lists to strings
    narrator_dialogue_str = ' '.join(narrator_dialogue_list)
    interviewer_dialogue_str = ' '.join(interviewer_dialogue_list)
    
    return narrator_dialogue_str, interviewer_dialogue_str

In [65]:
# not using this because this wouldnt improve the NN at all!
def _handle_multiple_narrators(text_content, file_name):
    # handle edge case of multiple narrators by duplicating metadata, separating
    # into two different rows
    # VAOHP0137
    # VAOHP0140
    # VAOHP0167
    if file_name in ["VAOHP0137", "VAOHP0140", "VAOHP0167"]:
        # identify if the initials belong the narrators or the interviewer
        # TODO: split text_content into two parts, one for each narrator
        # TODO: create two separate dictionaries for each narrator
        # TODO: return both dictionaries
        pass

    return text_content

In [66]:
def extract_interview_data_for_dataframe(text_content, file_name):
    """
    Extracts all interview-related data (metadata, initials, and dialogue) from the raw text
    of a PDF, suitable for populating a DataFrame.

    Args:
        text_content (str): The raw text content extracted from a PDF.
        file_name (str): The name of the PDF file.

    Returns:
        dict: A dictionary containing all extracted and derived fields.
    """

    # 0. Pre-processing: Normalize whitespace at beginning, Remove header content
    processed_text_content = re.sub(r'\s+', ' ', text_content).strip()
    processed_text_content, detected_language = _remove_filename_from_text(text_content, file_name)

    # 1. Extract initial metadata fields
    patterns = _get_metadata_patterns()
    metadata = _extract_initial_fields(processed_text_content, patterns)

    # Add detected language to metadata
    metadata['language'] = detected_language

    # Post-processing and refinement of metadata
    metadata['date'] = metadata['date']
    metadata['interview_length'] = metadata['interview_length']
    metadata['narrator_name'] = _refine_name_field(metadata['narrator_name'])
    metadata['interviewer_name'] = _refine_name_field(metadata['interviewer_name'])

    # 2. Derive initials using _extract_actual_speaker_initials
    actual_narrator_initials, actual_interviewer_initials = _extract_actual_speaker_initials(processed_text_content)
    metadata['narrator_initials'] = actual_narrator_initials
    metadata['interviewer_initials'] = actual_interviewer_initials

    # 3. Extract and parse dialogue
    dialogue_section = _extract_dialogue_section(processed_text_content)
    # Pass derived initials to the dialogue parser
    narrator_dialogue, interviewer_dialogue = _parse_speakers_dialogue(
        dialogue_section, metadata['narrator_initials'], metadata['interviewer_initials']
    )
    metadata['narrator_dialogue'] = narrator_dialogue
    metadata['interviewer_dialogue'] = interviewer_dialogue

    # Clean all string values in the final dictionary
    final_data = _clean_all_values(metadata)

    return final_data

In [80]:
def process_pdf_file(pdf_file, verbose=False):
  extracted_text = extract_text_from_pdf(pdf_file)
  extracted_interview_data = extract_interview_data_for_dataframe(extracted_text, pdf_file)
  if verbose:
    display(pdf_file)
    display("Extracted Interview Data:")
    for key, value in extracted_interview_data.items():
      display(f"  {key}: {value}")
  # Add the file name to the dictionary before returning
  match = re.search(r'(VA...\d+)', pdf_file)
  if match:
    extracted_interview_data['interview_id'] = match.group(1)
  else:
    extracted_interview_data['interview_id'] = np.nan
  return extracted_interview_data

# test to make sure it works
test = process_pdf_file(pdf_files[79])
for key, value in test.items():
  # use display on each key, value so that the dialogue doesn't take a bajillion
  # lines and output the entire transcript
  display(f"{key}: {value}")

'interview_id: VAOHP0102'

'narrator_name: Ysa Le'

'interviewer_name: Andy Le'

'date: November 10, 2012'

'location: Santa Ana, California'

...

'sub-collection: Vietnamese American Experience Course, Fall 2012'

'narrator_initials: YL'

'interviewer_initials: AL'

'narrator_dialogue: My name is Ysa Le. I am here with Andy, thank you for coming to VAALA to do this interview. I was born in 1970 in Saigon, Vietnam. I left Vietnam in 1983 with my family we went to France first because my grandfather worked for the French government and so we were able to immigrate to France first in 1983. It was after my dad was released from the reeducation camp. He got out in 1982 and so 1983 we were able to leave the country and we stayed in France from 1983 until 1985 and we immigrated to the United States after. I grew up in Saigon. I stayed most of my 12 years there and we tried to escape by boats in 1979, but we were not successful and I was actually got thrown in prison twice. My mom at that time was trying to take my two brothers and me to go by boat, and escape by boat. But we were Yeah a lot I think the first five years of my life was peaceful, that was from 1970-1975. I remember we would take trips to a resort area beach it’s in Saigon. We stay for a few

'interviewer_dialogue: Hello today is November 10, 2012. My name is Andy Le I am participating in the Vietnamese American Oral History Project. I am in the Vietnamese American Arts Letters Association with Ysa Le. If you can please introduce yourself. Thank you, let’s begin. Can you explain your date of birth and place of birth? Can you describe were you grew up and your hometown? Do you have any childhood memories you remember from Vietnam? So you mentioned your parents’ can you describe what your parents’ names was and how they were in Vietnam? Did you have any type of schooling in Vietnam? I was interested you said you immigrated to France, was there any particular language you have to pick up besides Vietnamese? Can you describe the Vietnamese community back then in France? Was it a large or small community? What memorable stories have your family members told you in the past? How did your parents, grandparents, and other relatives come to meet and marry? Was it Vietnam or the Unit

In [81]:
# attempt to load all files into dataframe
all_interview_data = []
failed_interview_files = []
failed_interview_data = []
df_saved_file_path = 'interview_data.xlsx'
df = None

try:
  df = pd.read_excel(df_saved_file_path)
  display(f"loading successful: {df.head()}")
except:
  display(f"{df_saved_file_path} not found, loading interview data into new file")
  for i, pdf_file in enumerate(pdf_files):
      print(f"Processing file {i+1}/{len(pdf_files)}: {pdf_file}")
      try:
        extracted_data = process_pdf_file(pdf_file)
        all_interview_data.append(extracted_data)
      except Exception as e:
        print(f"Error processing {pdf_file}: {e}")
        failed_interview_files.append(pdf_file)

  df = pd.DataFrame(all_interview_data)

  # Reorder columns to place 'interview_id' at the first position
  if 'interview_id' in df.columns:
    cols = ['interview_id'] + [col for col in df.columns if col != 'interview_id']
    df = df[cols]

  print("\nDataFrame created successfully. First 5 rows:")
  display(df.head())
  df.to_excel('interview_data.xlsx')

'interview_data.xlsx not found, loading interview data into new file'

Processing file 1/250: ./individual_transcripts/VAOHP0101_F01.pdf
Processing file 2/250: ./individual_transcripts/VAHF0002_F01.pdf
Processing file 3/250: ./individual_transcripts/VAOHP0111_F01.pdf
Processing file 4/250: ./individual_transcripts/VAOHP0069_F01.pdf
Error processing ./individual_transcripts/VAOHP0069_F01.pdf: failing due to bad initials narrator_initials = None	interviewer_initials = None
Processing file 5/250: ./individual_transcripts/VAHF0010_F01_Eng.pdf
Processing file 6/250: ./individual_transcripts/VAHF0012_F01_Viet.pdf
Processing file 7/250: ./individual_transcripts/VAOHP0100_F01_Eng.pdf
Processing file 8/250: ./individual_transcripts/VAOHP0058_F01_Viet.pdf
Processing file 9/250: ./individual_transcripts/VAOHP0215_F01.pdf
Processing file 10/250: ./individual_transcripts/VAOHP0074.2_F01_Eng.pdf
Processing file 11/250: ./individual_transcripts/VAOHP0050_F01_Eng.pdf
Processing file 12/250: ./individual_transcripts/VAOHP00377_F01.pdf
Processing file 13/250: ./individual_

,interview_id,narrator_name,interviewer_name,date,location,language,interview_length,sub-collection,narrator_initials,interviewer_initials,narrator_dialogue,interviewer_dialogue
0,VAOHP0101,Mary Hoang Long,Nina Mai Thi Long,"November 13, 2012","Westminster, California",English,01:32:16,"Vietnamese American Experience Course, Fall 2012",MHL,NL,Hi Nina. My full name is Mary Hoang Long. I am...,"Today is Tuesday, November 13, 2012. This is N..."
1,VAHF0002,Charlie Van Le,Pham Quang Tuan,"November 5, 2010","Westminster, California",English,00:12:13,Vietnamese American Heritage Foundation 500 Or...,CVL,PQT,"Right now, I just graduated from the Universit...",I’d like to ask you if you can state your name...
2,VAOHP0111,Alex Thai Nguyen,Samantha Erica Takahashi,"February 10, 2013","Westminster, California",English,01:19:24,"Linda Vo Class Oral Histories, 2013",ATN,SET,"Yes, my name is Alex Thai Nguyen. A-L-E-X Thai...","Today is Sunday, February 10, 2013. This is Sa..."
3,VAHF0010,Nguyễn Thị Hạnh Nhơn (Nthn),Nancy Bui/Trieu Giang (Tg) Date Of Interview: ...,NaN,NaN,English,01:23:22,Vietnamese American Heritage Foundation 500 Or...,NTHN,TG,"Yes, my dear . My name is Nguyễn thi Hanh Nhon...",Mrs Nguyễn Hạnh Nhơn . Can you let us know whe...
4,VAHF0012,Nguyễn Chí Thiện (Nct),Nancy Bùi/Triều Giang (Tg),"November 4, 2010","Westminster, California",Vietnamese,01:32:25,Vietnamese American Heritage Foundation 500 Or...,NCT,TG,"Tôi tên là Nguyễn Chí Thiện , tôi đẻ ở Hà Nội ...",Xin ông cho biết quý danh và ông sinh trưởng ở...


/var/folders/g2/6ym21yr10t96b5wj2y6qr4g40000gn/T/ipykernel_32163/4160527589.py:31: UserWarning: Cell contents too long (37674), truncated to 32767 characters
  df.to_excel('interview_data.xlsx')
/var/folders/g2/6ym21yr10t96b5wj2y6qr4g40000gn/T/ipykernel_32163/4160527589.py:31: UserWarning: Cell contents too long (36350), truncated to 32767 characters
  df.to_excel('interview_data.xlsx')
/var/folders/g2/6ym21yr10t96b5wj2y6qr4g40000gn/T/ipykernel_32163/4160527589.py:31: UserWarning: Cell contents too long (59119), truncated to 32767 characters
  df.to_excel('interview_data.xlsx')
/var/folders/g2/6ym21yr10t96b5wj2y6qr4g40000gn/T/ipykernel_32163/4160527589.py:31: UserWarning: Cell contents too long (82596), truncated to 32767 characters
  df.to_excel('interview_data.xlsx')
/var/folders/g2/6ym21yr10t96b5wj2y6qr4g40000gn/T/ipykernel_32163/4160527589.py:31: UserWarning: Cell contents too long (51886), truncated to 32767 characters
  df.to_excel('interview_data.xlsx')
/var/folders/g2/6ym21yr10

In [84]:
display(df)

,interview_id,narrator_name,interviewer_name,date,location,language,interview_length,sub-collection,narrator_initials,interviewer_initials,narrator_dialogue,interviewer_dialogue
0,VAOHP0101,Mary Hoang Long,Nina Mai Thi Long,"November 13, 2012","Westminster, California",English,01:32:16,"Vietnamese American Experience Course, Fall 2012",MHL,NL,Hi Nina. My full name is Mary Hoang Long. I am...,"Today is Tuesday, November 13, 2012. This is N..."
1,VAHF0002,Charlie Van Le,Pham Quang Tuan,"November 5, 2010","Westminster, California",English,00:12:13,Vietnamese American Heritage Foundation 500 Or...,CVL,PQT,"Right now, I just graduated from the Universit...",I’d like to ask you if you can state your name...
2,VAOHP0111,Alex Thai Nguyen,Samantha Erica Takahashi,"February 10, 2013","Westminster, California",English,01:19:24,"Linda Vo Class Oral Histories, 2013",ATN,SET,"Yes, my name is Alex Thai Nguyen. A-L-E-X Thai...","Today is Sunday, February 10, 2013. This is Sa..."
3,VAHF0010,Nguyễn Thị Hạnh Nhơn (Nthn),Nancy Bui/Trieu Giang (Tg) Date Of Interview: ...,NaN,NaN,English,01:23:22,Vietnamese American Heritage Foundation 500 Or...,NTHN,TG,"Yes, my dear . My name is Nguyễn thi Hanh Nhon...",Mrs Nguyễn Hạnh Nhơn . Can you let us know whe...
4,VAHF0012,Nguyễn Chí Thiện (Nct),Nancy Bùi/Triều Giang (Tg),"November 4, 2010","Westminster, California",Vietnamese,01:32:25,Vietnamese American Heritage Foundation 500 Or...,NCT,TG,"Tôi tên là Nguyễn Chí Thiện , tôi đẻ ở Hà Nội ...",Xin ông cho biết quý danh và ông sinh trưởng ở...
...,...,...,...,...,...,...,...,...,...,...,...,...
225,VAHF0007,Andrew Tuan Pham (Aka Pham Quan Tuan),Roger Minh Le,"November 5, 2010","Westminster, California",English,00:21:12,Vietnamese American Heritage Foundation 500 Or...,PQT,RL,My name is Andrew Tuan Pham. But I also go by ...,"Today is November 5, 2010. We are here in West..."
226,VAOHP0114,Steve Pham,David Liu,"February 23, 2013","Garden Grove, California",English,01:52:53,Linda Vo Class Oral Histories 2013,SP,DL,"Steve Pham 8/18/1960 Saigon, Vietnam My father...",This is David Liu with the Vietnamese American...
227,VAOHP0084,Keith Ky Thanh Van,Thuý Võ Đặng,"August 8, 2012","Huntington Beach, California",English,01:36:36,Thuy Vo Dang Oral Histories,KV,TVD,"My name is Keith Van, full name is Keith Kỳ Th...","First please introduce your name, birthdate an..."
228,VAOHP0079,Kiên Tâm Nguyễn,Thúy Võ Đặng,"May 22, 2012","Fountain Valley, California Sub-­‐Collection: ...",English,NaN,NaN,KTN,TVD,Ok. My name is Kien Tam Nguyen. I was born on ...,"First of all, could you please tell us your na..."


In [85]:
display(failed_interview_files)

['./individual_transcripts/VAOHP0069_F01.pdf',
 './individual_transcripts/VAOHP0002_F01.pdf',
 './individual_transcripts/VAOHP0359_F01_Eng.pdf',
 './individual_transcripts/VAOHP0068_F01.pdf',
 './individual_transcripts/VAOHP0035_F01.pdf',
 './individual_transcripts/VAOHP0353_F01.pdf',
 './individual_transcripts/VAOHP0137_F01_Eng.pdf',
 './individual_transcripts/VAOHP0097_F01.pdf',
 './individual_transcripts/VAOHP0359_F01_Viet.pdf',
 './individual_transcripts/VAOHP0036_F01.pdf',
 './individual_transcripts/VAOHP0089_F01_Viet.pdf',
 './individual_transcripts/VAOHP0010_F01.pdf',
 './individual_transcripts/VAOHP0376_F01.pdf',
 './individual_transcripts/VAOHP0349_F01.pdf',
 './individual_transcripts/VAOHP0096_F01.pdf',
 './individual_transcripts/VAOHP0363_F01.pdf',
 './individual_transcripts/VAOHP0040.1_F01.pdf',
 './individual_transcripts/VAOHP0167_F01.pdf',
 './individual_transcripts/VAOHP0091_F01.pdf',
 './individual_transcripts/VAOHP0040.2_F01.pdf']

In [75]:
empty_dialogues_narrator = (df["narrator_dialogue"].isnull() | (df["narrator_dialogue"] == "")).sum()
print(f"Number of empty rows in 'narrator_dialogue': {empty_dialogues_narrator}")
empty_dialogues_interviewer = (df["interviewer_dialogue"].isnull() | (df["interviewer_dialogue"] == "")).sum()
print(f"Number of empty rows in 'interviewer_dialogue': {empty_dialogues_interviewer}")

Number of empty rows in 'narrator_dialogue': 0
Number of empty rows in 'interviewer_dialogue': 2


In [82]:
display(extract_text_from_pdf('individual_transcripts/VAOHP0002_F01.pdf'))

' VAOHP0002  Vietnamese American History Project, UC Irvine Narrator: NICOLE NGUYEN Interview: Malessa Tem Date: February 26, 2012 Location: Yorba Linda, California Sub-Collection: Vietnamese American Experience Winter 2012 Length of Interview: 00:85:50 MT: My name is Malessa Tem and today is February 26, 2012. I am here with Nicole in Yorba Linda. Nicole, can you tell us more about more about your name, and your date of birth, and your place of birth? NN: My name is Nicole Thu Nga Nguyen. I am thirty-five and I was born on November 16, 1976 in Da Nang, Vietnam. MT: Do you remember anything about your place of birth? NN: No, actually. None at all. MT: Okay, what are some of your early memories? NN: My childhood memories have to deal with going around picking up plastics and plastic bags, anything plastics. So I can sell to make some money and buy some candy. Going to school. Pre-school, and you know playing hopscotch with my neighbors and friends. Pretty much hanging just around the ho

In [68]:
tester_text = extract_text_from_pdf(pdf_files[3])
display(_extract_dialogue_section(tester_text))


''

In [ ]:
_extract_actual_speaker_initials(tester_text)

In [54]:
pdf_files[3]

'./individual_transcripts/VAOHP0069_F01.pdf'